# Notebook 02: Đặt Câu hỏi Nghiên cứu và Phân tích Chuyên sâu (Non-ML Questions Q1–Q5)
**Đề tài:** Phân tích Sự cố Bất lợi Thuốc Thú y (OpenFDA Animal & Veterinary Adverse Events) giai đoạn 2025 và Đối chiếu Xu hướng 2026 (Q1–Q2)

**Môn học:** CSC17104 – Lập trình cho Khoa học Dữ liệu  
**Nhóm:** Prg4DS-G3

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.preprocessing import log_row_count_change
from src.visualization import set_project_plot_style
set_project_plot_style()

---
## 1. Tổng quan Hệ thống Câu hỏi Nghiên cứu & Ma trận Độ sâu (Mục 3.3)

Theo yêu cầu của môn học, nhóm 3 thành viên xây dựng **$2 \times 3 = 6$ câu hỏi** (5 câu hỏi phân tích thống kê/khám phá chuyên sâu trong Notebook này và 1 câu hỏi Học máy trong `03_ml_question.ipynb`). Mỗi câu hỏi Non-ML đáp ứng **ít nhất 2 tiêu chí độ sâu (D1–D6)** và toàn bộ bộ câu hỏi bao phủ cả **6/6 tiêu chí (D1–D6)**:

| Mã | Chủ sở hữu | Câu hỏi nghiên cứu | Tiêu chí độ sâu (D1–D6) |
|---|---|---|---|
| **Q1** | **M1 — Khang** | Những phản ứng bất lợi (VeDDRA terms) nào được báo cáo nhiều nhất ở Chó, Mèo và các loài khác trong năm 2025, và xu hướng biến động theo quý (kèm dấu hiệu mùa vụ trong phạm vi 1 năm) của chúng diễn biến ra sao khi bước sang nửa đầu năm 2026 (Q1–Q2)? | **D1** (Combining tables), **D2** (Reshaping & time series), **D5** (Quantifying uncertainty) |
| **Q2** | **M2 — Nhi** | Tỷ lệ sự cố nghiêm trọng (`serious_ae`) và kết cục tử vong trong năm 2025 thay đổi thế nào theo giai đoạn tuổi (`life_stage`), cân nặng chuẩn hóa (`weight_kg`) và giới tính sau khi kiểm soát biến nhiễu là **loài** và **đa trị liệu (`num_drugs`)**? | **D3** (Derived variables), **D4** (Controlling for confounder), **D5** (CI/Hypothesis test), **D6** (Robustness check) |
| **Q3** | **M3 — Kiệt** | Các hoạt chất (`active_ingredients`) và đường dùng (`route`) nào có tỷ suất đồng xuất hiện (Reporting Odds Ratio) đặc trưng nhất với từng nhóm phản ứng cụ thể, và cấu trúc này khác biệt ra sao giữa **Safety Issue** và **Lack Of Expected Effectiveness** trong năm 2025? | **D1** (Multi-table merge), **D2** (Pivot/melt matrix), **D4** (Subgroup control), **D5** (Odds Ratio & 95% CI) |
| **Q4** | **M3 — Kiệt** | Việc dùng thuốc sai nhãn (`off-label`) kết hợp với đa trị liệu (`polypharmacy`) trong năm 2025 có liên hệ như thế nào với mức độ nghiêm trọng (`serious_ae`) và kết cục (`outcome`) sau khi kiểm soát biến nhiễu là **loài** và **đối tượng báo cáo (`primary_reporter`)**? | **D1** (Combining tables), **D3** (Derived indicators), **D4** (Controlling for confounder), **D6** (Robustness check) |
| **Q5** | **M2 — Nhi** | Giữa Bác sĩ thú y (`Veterinarian`), Chủ vật nuôi (`Animal Owner`) và Nhân viên y tế khác, có sự khác biệt có ý nghĩa thống kê nào về **độ trễ báo cáo (`reporting_delay_days`)**, loại sự kiện và tỷ lệ nghiêm trọng qua 4 quý năm 2025 và 2026 Q1–Q2 không? | **D2** (Period-over-period & lag), **D3** (Derived delay variable), **D5** (Kruskal-Wallis / Bootstrap CI), **D6** (Outlier window robustness) |
| **Q6 (ML)** | **M1 — Khang** | *(Thực hiện tại `03_ml_question.ipynb`)* Huấn luyện mô hình trên dữ liệu 2025 (Q1–Q4) để dự đoán khả năng một báo cáo trong năm 2026 (Q1–Q2) là sự cố nghiêm trọng (`serious_ae`) dựa trên thông tin ban đầu lúc tiếp nhận. | **ML Requirements 1–6** (Time-based split 2025 Q1–Q4 $\rightarrow$ 2026 Q1–Q2, Pipeline, Leakage-free, Error & Feature Importance) |

---
## 2. Câu hỏi Q1 (Owner: Đinh Công Khang — M1)
### Phát biểu câu hỏi & Động lực (Question & Motivation)
- **Câu hỏi:** Những phản ứng bất lợi (VeDDRA terms) nào chiếm tỷ trọng cao nhất ở Chó, Mèo và các loài gia súc/vật nuôi khác trong năm 2025, và xu hướng theo quý (kèm dấu hiệu mùa vụ trong phạm vi 1 năm) của các phản ứng hàng đầu này thay đổi như thế nào cho đến nửa đầu năm 2026 (Q1–Q2)?
- **Người quan tâm & Quyết định hỗ trợ:** Bác sĩ thú y lâm sàng và nhà sản xuất thuốc thú y cần biết đặc thù phản ứng theo từng loài (ví dụ: nôn mửa/lờ đờ ở chó mèo hay kém hiệu quả diệt ký sinh trùng theo mùa) để tư vấn theo dõi sau dùng thuốc theo mùa vụ.
- **Tiêu chí độ sâu áp dụng:** `D1` (Kết hợp bảng `reports` và `reactions`), `D2` (Resampling theo quý `2025Q1`–`2026Q2`, rolling average, pivot table), `D5` (Khoảng tin cậy Bootstrap 95% và kiểm định $\chi^2$).

### 2.1. Tiền xử lý cho Q1 (Preprocessing & Row Count Tracking)
*(Mô tả các bước lọc/gộp bằng Markdown và in số dòng trước/sau mỗi bước bằng `log_row_count_change`)*

In [ ]:
# TODO [Giai đoạn 3 - Khang]: Tiền xử lý, phân tích D1+D2+D5 và vẽ >= 2 biểu đồ cho Q1


---
## 3. Câu hỏi Q2 (Owner: Trần Ngô Uyên Nhi — M2)
### Phát biểu câu hỏi & Động lực (Question & Motivation)
- **Câu hỏi:** Tỷ lệ sự cố nghiêm trọng (`serious_ae`) và kết cục tử vong trong năm 2025 thay đổi như thế nào theo giai đoạn tuổi (`life_stage`), cân nặng chuẩn hóa (`weight_kg`) và giới tính của vật nuôi sau khi kiểm soát biến nhiễu là **loài** và **số lượng thuốc dùng đồng thời (`num_drugs`)**?
- **Người quan tâm & Quyết định hỗ trợ:** Bác sĩ thú y cần xác định nhóm vật nuôi nguy cơ cao (ví dụ thú non hoặc thú già, nhẹ cân, đang dùng đa trị liệu) để điều chỉnh liều lượng và theo dõi sát.
- **Tiêu chí độ sâu áp dụng:** `D3` (Biến dẫn xuất `age_years`, `weight_kg`, `life_stage`, `num_drugs`), `D4` (Kiểm soát biến nhiễu theo loài & đa trị liệu), `D5` (Kiểm định thống kê & khoảng tin cậy 95%), `D6` (Kiểm tra độ vững khi thay đổi quy tắc cắt ngưỡng ngoại lai tuổi/cân nặng).

### 3.1. Tiền xử lý cho Q2 (Preprocessing & Row Count Tracking)

In [ ]:
# TODO [Giai đoạn 3 - Nhi]: Tiền xử lý, phân tích D3+D4+D5+D6 và vẽ >= 2 biểu đồ cho Q2


---
## 4. Câu hỏi Q3 (Owner: Huỳnh Tuấn Kiệt — M3)
### Phát biểu câu hỏi & Động lực (Question & Motivation)
- **Câu hỏi:** Các hoạt chất (`active_ingredients`) và đường dùng (`route`) nào thường đi kèm với những reaction cụ thể, và cấu trúc đồng xuất hiện (co-occurrence pattern / Reporting Odds Ratio) này khác nhau như thế nào giữa nhóm **Safety Issue** và **Lack Of Expected Effectiveness** trong năm 2025?
- **Người quan tâm & Quyết định hỗ trợ:** Chuyên gia cảnh giác dược (FDA CVM) và bộ phận R&D dược thú y cần phân biệt rõ hoạt chất nào gây độc tính sinh học (Safety Issue) so với hoạt chất nào đang bị giảm hiệu lực/kháng thuốc (Lack of Efficacy, như thuốc trị giun tim hoặc bọ chét).
- **Tiêu chí độ sâu áp dụng:** `D1` (Kết hợp 3 bảng `reports`, `drugs`, `reactions` có kiểm chứng khóa), `D2` (Reshape ma trận đồng xuất hiện & so sánh giai đoạn), `D4` (Kiểm soát theo loài), `D5` (Reporting Odds Ratio kèm khoảng tin cậy 95%).

### 4.1. Tiền xử lý cho Q3 (Preprocessing & Row Count Tracking)

In [ ]:
# TODO [Giai đoạn 3 - Kiệt]: Tiền xử lý, phân tích D1+D2+D4+D5 và vẽ >= 2 biểu đồ cho Q3


---
## 5. Câu hỏi Q4 (Owner: Huỳnh Tuấn Kiệt — M3)
### Phát biểu câu hỏi & Động lực (Question & Motivation)
- **Câu hỏi:** Việc dùng thuốc đúng nhãn (`used_according_to_label = true`) so với dùng sai nhãn (`off-label`) trong năm 2025 có liên hệ như thế nào với mức độ nghiêm trọng (`serious_ae`) và kết cục (`outcome`) sau khi kiểm soát biến nhiễu là **loài vật nuôi** và **đối tượng báo cáo (`primary_reporter`)**?
- **Người quan tâm & Quyết định hỗ trợ:** Cơ quan quản lý dược thú y và phòng khám cần bằng chứng định lượng về rủi ro của việc kê đơn off-label hoặc tự ý dùng thuốc trái hướng dẫn trên từng loài.
- **Tiêu chí độ sâu áp dụng:** `D1` (Kết hợp `reports`, `drugs`, `outcomes`), `D3` (Biến dẫn xuất cấp báo cáo `has_off_label_drug`, `worst_outcome_rank`), `D4` (Kiểm soát biến nhiễu `species` và `primary_reporter`), `D6` (Kiểm tra độ vững khi loại bỏ vs. giữ nhóm khuyết thông tin `used_according_to_label`).

### 5.1. Tiền xử lý cho Q4 (Preprocessing & Row Count Tracking)

In [ ]:
# TODO [Giai đoạn 3 - Kiệt]: Tiền xử lý, phân tích D1+D3+D4+D6 và vẽ >= 2 biểu đồ cho Q4


---
## 6. Câu hỏi Q5 (Owner: Trần Ngô Uyên Nhi — M2)
### Phát biểu câu hỏi & Động lực (Question & Motivation)
- **Câu hỏi:** Người báo cáo (`Veterinarian`, `Animal Owner`, `Other Health Care Professional`...) có sự khác biệt có ý nghĩa thống kê nào về **độ trễ gửi báo cáo (`reporting_delay_days = original_receive_date - onset_date`)**, loại sự kiện (`type_of_information`) và mức độ nghiêm trọng (`serious_ae`) qua 4 quý năm 2025 và 2026 Q1–Q2 hay không?
- **Người quan tâm & Quyết định hỗ trợ:** Đội ngũ vận hành hệ thống thu thập dữ liệu của FDA CVM cần hiểu hành vi báo cáo và độ trễ của từng nhóm đối tượng để cải thiện quy trình tiếp nhận và hiệu chỉnh độ trễ khi phân tích dữ liệu thời gian thực (như dữ liệu mới của năm 2026).
- **Tiêu chí độ sâu áp dụng:** `D2` (So sánh chuỗi thời gian và độ trễ theo kỳ), `D3` (Biến dẫn xuất `reporting_delay_days`), `D5` (Kiểm định phi tham số Kruskal-Wallis / Mann-Whitney U và Bootstrap CI cho trung vị độ trễ), `D6` (Kiểm tra độ vững với các ngưỡng cắt độ trễ cực đoan 180 ngày vs. 365 ngày).

### 6.1. Tiền xử lý cho Q5 (Preprocessing & Row Count Tracking)

In [ ]:
# TODO [Giai đoạn 3 - Nhi]: Tiền xử lý, phân tích D2+D3+D5+D6 và vẽ >= 2 biểu đồ cho Q5
